In [1]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast
import pandas as pd
import numpy
import os

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.7.1+cu118
Cuda version: 11.8
transformers version: 5.3.0
GPU 사용 가능여부: True


### SFT 데이터 셋

In [2]:
import json
data_path_1_SFT = os.getenv('HOME') + '/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

### RM 데이터 셋

In [3]:
data_path_2_RM = os.getenv('HOME') + '/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

### PPO 데이터 셋

In [4]:
data_path_3_PPO = os.getenv('HOME') + '/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

## 평가 데이터 셋


### 정량적 평가 
- chat model의 성능을 평가하기 위해, 두 답변 중 더 선호되는 답변을 인간이 선택한 Korean-Human-Judgemets dataset을 활용했다.
- dataset으로부터 chosen 답변을 reference로 선정하고, chat model이 생성한 답변과의 BLEU를 계산하여 성능을 평가한다.

In [5]:
import datasets
import pandas as pd

test_data = datasets.load_dataset("HAERAE-HUB/Korean-Human-Judgements")
test_df = test_data['train'].to_pandas()
test_df.head()

,instruction,response_a,response_b,decision,source
0,한국의 스타크래프트 브루드워 게임 대회의 역대 우승자를 알려줘,한국의 스타크래프트 브루드워 게임 대회의 역대 우승자는 다음과 같습니다:\n\n1....,한국의 스타크래프트 브루드워 게임 대회의 역대 우승자는 다음과 같습니다.\n\n-...,B,instructkr/ko-chatbot-arena-leaderboard
1,인터넷 커뮤니티 게시글의 문체로 삼성과 애플의 스마트폰을 비교하는 글을 써줘.,안녕하세요 여러분! 애플과 삼성의 스마트폰을 비교하는 것에 대해 이야기해 보고 싶었...,안녕하세요 여러분~ 오늘은 삼성과 애플의 스마트폰 중에서 더 좋은 것은 哪个인지 사...,A,instructkr/ko-chatbot-arena-leaderboard
2,회사를 때려치고 싶을땐 어떻게 해야할까? 대표를 때려도 되나?,회사를 때려치고 싶다면 다음과 같은 방법을 고려해 보세요:\n\n1. 문제 파악: ...,회사를 때려치고 싶을 때는 법적으로 가능한 경우가 많지 않습니다. 또한 사업체 소유...,A,instructkr/ko-chatbot-arena-leaderboard
3,오른쪽으로 기울어져 있는 책상에서 완벽한 구의 형태인 공을 왼쪽으로 살짝 밀었을때 ...,"책상이 오른쪽으로 기울어져 있다면, 왼쪽으로 살짝 밀었을 때 공은 책상의 기울기를...",오른쪽으로 기울어져 있는 책상 위에서 완벽한 구의 형태인 공을 왼쪽으로 살짝 밀었을...,Tie,instructkr/ko-chatbot-arena-leaderboard
4,시각적이고 청각적 미각을 자극하도록 시골에 보이는 작은 집을 묘사해주세요.,한적한 시골 마을에 자리한 작고 아늑한 집은 마치 동화 속에서 나온 것처럼 보입니다...,"작고 조그마한 집은 밝은 분홍색 벽돌로 지어져 있으며, 지붕은 골판지로 덮여 있습니...",Tie,instructkr/ko-chatbot-arena-leaderboard


In [6]:
test_df = test_df[test_df['decision'] != 'Tie']

In [7]:
import numpy as np
test_df["chosen"] = np.where(
    test_df["decision"] == 'A',
    test_df["response_a"],
    test_df["response_b"]
)

## foundation model 변경
-  skt/ko-gpt-trinity-1.2B-v0.5 모델로 변경하여 실험 진행
-  학습 파라미터 수를 줄이기 위해 LoRA 사용

## Supervised Fine-Tuning

In [8]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

In [9]:
!pip install peft

In [10]:
!pip install bitsandbytes

In [41]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,
)

print(tokenizer)

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

TokenizersBackend(name_or_path='skt/kogpt2-base-v2', vocab_size=51200, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<usr>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<sys>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	7: AddedT

In [42]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
kogpt-2_tokens,▁바람,도,▁없는,▁공중에,▁수직,의,▁파,문을,▁내,이며,▁고,요,히,▁떨어지는,▁오동,잎은,▁누,구의,▁발자,취,▁입,니까,.
Input_IDs,10891,7235,9712,49207,14438,8143,9203,9941,9094,9639,9065,8084,8811,21215,34769,19985,9669,10139,21626,8408,9241,23775,389


In [43]:

class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [44]:
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [45]:
train_dataset = SFT_dataset(data_path_1_SFT=os.getenv('HOME') +'/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

input : tensor([  739,   378,   378,   378, 14659, 13394, 37091, 10651,   383, 25841,
         8006, 14914,   375,  7673, 20479,  8091, 22311,  9036, 30902, 13675,
          375,   378,   378,   378, 41951,   454,  9549, 20549,   383,  8142,
         7192, 14914,   382, 37767, 13753,  8263,  7166,   739,  8352,  7659,
         9594, 25585, 13600,  8022,  9378, 11532,  9887, 11218,  9111, 16691,
        10351, 10561,  9128, 20479,  8091,  9065,  9446,  9036, 28420, 26521,
        10163, 26367,  6958,  9030,  9882, 12317, 25882,  9209, 37194, 10351,
         9036, 12168, 10529, 15989,  9719, 15434, 10552, 11188, 13362,  9036,
        15805, 11300, 11846,  9146, 16691,  9181,  7397, 15806, 13480, 11342,
        17596,  9161, 19996,  9025, 25006, 18595,  9966, 12592, 10751, 11814,
         8711,  9046, 12450,  9117,  7377, 12521,     1])
output: tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -10

In [28]:
training_args = transformers.TrainingArguments(
    output_dir="test",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16 = True
    )
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

In [29]:
trainer.train()
model.save_pretrained('models/output_1_SFT')

Step,Training Loss
500,2.977753
1000,2.783816
1500,2.685269


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [46]:
generator = transformers.pipeline('text-generation', model='models/output_1_SFT', tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'max_new_tokens', 'repetition_penalty', 'no_repeat_ngram_size', 'num_beams', 'top_k', 'early_stopping', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer t


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 고기를 먹을 수 없습니다. 하지만 일반적으로 불고기용 고기는 건강에 좋은 음식 중 하나입니다. 쇠고기의 영양가가 높아 건강에도 좋기 때문입니다. 따라서 고기를 먹는 것은 매우 중요합니다.辛丑)辛丑申丑申丑辛丑辛丑辰丑

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 41대 부통령직을 수행했습니다.恩恩恩恩殷恩恩恩銀恩恩恩은 1951년부터 1952년까지 부통령직을 수행하였습니다. 이 기간 동안 그는 미국 역사상 가장 큰 대통령 중 한 명이었습니다.恩隱恩恩恩)은 1954년부터 1964년까지 부통령직을 맡았

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고는 미국 캘리포니아주 로스앤젤레스에 위치한 도시입니다.高橋라고도 불립니다.高橋)은 시카고에서 가장 오래된 도시 중 하나입니다.鼓橋는 시카고에서 가장 유명한 도시 중 하나이기도 합니다.鼓

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'저는 인공지능 챗봇이기 때문에 미세먼지에 대한 정보를 알 수 없습니다. 하지만, 미세먼지는 건강에 매우 중요한 요소 중 하나입니다. 건강한 생활습관을 유지하는 것이 중요합니다. 또한, 미세먼지가 심한 날에는 마스크를 착용하여 미세먼지를 차단하는 것이 좋습니다.了


## SFT - BLEU

In [47]:
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

list_prompt = test_df['instruction'][:100]

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)



Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

In [48]:
idx = 0
total_bleu_score = 0.0
generate_sentence = []
for i, (prompt, result) in tqdm(enumerate(zip(list_prompt, list_result))):
    generate_sentence.append(result[0]['generated_text'])
    reference = tokenizer(test_df['chosen'].iloc[idx]).tokens()
    gen_sen = result[0]['generated_text'][result[0]['generated_text'].index('Response(응답):')+13:]
    candidate = tokenizer(gen_sen).tokens()
    
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=SmoothingFunction().method1)
    total_bleu_score += bleu_score
    if i < 5:
        print()
        print(result[0]['generated_text'])
    idx+=1
print('Average BLEU score',total_bleu_score / idx)

28it [00:00, 222.21it/s]


### Instruction(명령어):
한국의 스타크래프트 브루드워 게임 대회의 역대 우승자를 알려줘

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만, 한국 프로게이머들의 대회 사상 우승자는 다음과 같습니다.\n\n1. KOREA (Korean League of the Esports)\n2. NC 다이노스(N

### Instruction(명령어):
인터넷 커뮤니티 게시글의 문체로 삼성과 애플의 스마트폰을 비교하는 글을 써줘.

### Response(응답):'저는 인공지능 어시스턴트이기 때문에, 인터넷 커뮤니티에서 사용되는 글들이 어떤 문맥에서 작성되었는지 알 수 없습니다. 하지만, 인터넷 커뮤니티에서 사용하는 글들은 대체로 다음과 같습니다.\n\n1. 삼성 갤럭시 노트\n2. 애플 아이패드\n3. 애플 아이

### Instruction(명령어):
회사를 때려치고 싶을땐 어떻게 해야할까? 대표를 때려도 되나?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 회사의 상황에 대한 정보를 알 수 없습니다. 하지만 일반적으로 회사에서는 대표이사를 때려치고 싶은 이유가 있을 수 있습니다. 이를 위해 다음과 같은 방법을 시도해 볼 수 있습니다.\n\n1. 경영전략 수립: 회사가 어떤 방향으로 나아갈지에 따라 다르지만, 중요한 것은 경영전략

### Instruction(명령어):
반말로 말하자. 그나저나 질문 하나 할게. 다층 퍼셉트론이 뭐야?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 답변을 드리기 어렵습니다. 하지만 일반적으로 반말을 사용하는 사람들은 다음과 같습니다.\n\n1. "반말": 반말을 사용하는 것은 자연스러운 표현입니다. 예를 들어, "안녕하세요", "미안해요", "죄송합니다"

### Instruction(명령어):
예수금이랑 인출가능금액 차이나는 이유가 뭔가요?제가 신한 해외 주식 소수점 7천원대를 2개 매수 신청했다가 미체결 됐는데요

100it [00:00, 333.29it/s]

Average BLEU score 0.00229137119020834


## SFT -  ko-gpt-trinity-1.2B-v0.5

In [40]:
torch.cuda.empty_cache()

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "skt/ko-gpt-trinity-1.2B-v0.5"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name,  bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
kogpt-2_tokens,▁바람,도,▁없는,▁공,중에,▁수직,의,▁파,문을,▁내,이며,▁고요,히,▁떨어지는,▁오,동,잎,은,▁누구,의,▁발자,취,▁입,니까.
Input_IDs,31140,20780,30359,30016,31373,41427,25792,30163,31047,30024,31111,51068,29936,36152,30027,20801,25846,25768,31199,25792,44202,27472,30148,37708


### LoRA 설정

In [8]:
for name, module in model.named_modules():
    print(name)


transformer
transformer.wte
transformer.wpe
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.c_attn
transformer.h.0.attn.c_proj
transformer.h.0.attn.attn_dropout
transformer.h.0.attn.resid_dropout
transformer.h.0.ln_2
transformer.h.0.mlp
transformer.h.0.mlp.c_fc
transformer.h.0.mlp.c_proj
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.c_attn
transformer.h.1.attn.c_proj
transformer.h.1.attn.attn_dropout
transformer.h.1.attn.resid_dropout
transformer.h.1.ln_2
transformer.h.1.mlp
transformer.h.1.mlp.c_fc
transformer.h.1.mlp.c_proj
transformer.h.1.mlp.act
transformer.h.1.mlp.dropout
transformer.h.2
transformer.h.2.ln_1
transformer.h.2.attn
transformer.h.2.attn.c_attn
transformer.h.2.attn.c_proj
transformer.h.2.attn.attn_dropout
transformer.h.2.attn.resid_dropout
transformer.h.2.ln_2
transformer.h.2.mlp
transformer.h.2.mlp.c_fc
transformer.h.2.mlp

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                      # rank
    lora_alpha=16,
    target_modules=["c_attn"],  # 중요!! 모델의 모듈명과 구조를 확인해 lora를 어디에 적용할 것인지 판단
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [10]:
model = get_peft_model(model, lora_config)
#model.add_adapter(lora_config)
model.print_trainable_parameters()

trainable params: 1,474,560 || all params: 1,164,030,720 || trainable%: 0.1267


In [14]:
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [14]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [15]:

training_args = transformers.TrainingArguments(
    output_dir="test_trinity",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    optim="paged_adamw_8bit"  # 중요
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

In [16]:
trainer.train()
model.save_pretrained('models/output_1_SFT_trinity')

/opt/conda/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.400222
20,3.082248
30,2.815555
40,2.798692
50,2.732888
60,2.669337
70,2.758440
80,2.699789
90,2.586947
100,2.676126


/opt/conda/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.12/site-packages/torch/_dynamo/

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "skt/ko-gpt-trinity-1.2B-v0.5"


tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name,  bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,)


generator = transformers.pipeline('text-generation', model='models/output_1_SFT_trinity', tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/opt/conda/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: models/output_1_SFT_trinity
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0...23}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0...23}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
transformer.h.{0...23}.attn.c_attn.lora_B.default.weight                  | MISSING    | 
transformer.h.{0...23}.attn.c_attn.lora_A.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'no_repeat_ngram_size', 'top_k', 'repetition_p


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답): Could you tell me about your beef?
 (고기를 좀 더 주실 수 있나요?)
 What can I have to do for lunch?
 (점심은 무엇을 드시겠습니까?)
 Do you have anything to eat?


### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):Responsibility(책임감)
 ### Recognition(칭찬):Communication(커뮤니케이션)
 ### Service(봉사):Services(서비스)
 ### Experience(경험):Experiences(경험)


### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답): How can I get to the airport? (공항까지 어떻게 갈 수 있어?)
 ## Explain(설명): What does it look like? (무엇이 그런 것 같아?)
 ## Describe(추측): What's the

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답): 오늘 미세먼지 어때?
  예시 답안: 미세먼지
  본문 스크립트: 
  정답 확인: 
 [이미지 출처] - 네이버 지식백과 - 다음 지식백과 - 카카오 브런치 - 유튜브
  이미지 출처: Pixabay
  오늘의 퀴


### SFT - ko-gpt-trinity BLEU 계산

In [21]:
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

list_prompt = test_df['instruction'][:100]

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)



Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

ValueError: not enough values to unpack (expected 3, got 2)

In [48]:
idx = 0
total_bleu_score = 0.0
generate_sentence = []
for i, (prompt, result) in tqdm(enumerate(zip(list_prompt, list_result))):
    generate_sentence.append(result[0]['generated_text'])
    reference = tokenizer(test_df['chosen'].iloc[idx]).tokens()
    gen_sen = result[0]['generated_text'][result[0]['generated_text'].index('Response(응답):')+13:]
    candidate = tokenizer(gen_sen).tokens()
    
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=SmoothingFunction().method1)
    total_bleu_score += bleu_score
    if i < 5:
        print()
        print(result[0]['generated_text'])
    idx+=1
print('Average BLEU score',total_bleu_score / idx)

30it [00:00, 257.95it/s]


### Instruction(명령어):
한국의 스타크래프트 브루드워 게임 대회의 역대 우승자를 알려줘

### Response(응답):게임이 종료되었음을 알림
 ## Successful(성공):우승자에게 상금을 줌
 ## Excellent(훌륭함):우승자들에게 상금을 지급하지 않음
 ## Disappointed(실망):상금을 지급함
 ## Failure

### Instruction(명령어):
인터넷 커뮤니티 게시글의 문체로 삼성과 애플의 스마트폰을 비교하는 글을 써줘.

### Response(응답):Response.response는 response와 responce를 합친 말로, response, responce, reponse가 합쳐진 단어다.
 인터넷 커뮤니티 게시글에 댓글을 다는 방법
 인터넷 커뮤니티 게시글을 자신의 블로그나 미니홈피

### Instruction(명령어):
회사를 때려치고 싶을땐 어떻게 해야할까? 대표를 때려도 되나?

### Response(응답):대표를 때려치우고 싶을 땐 어떻게 해야할까?
 ###
 How to quit the company? 회사를 그만두고 싶을때 어떻게 해야하는가?
 ###
 What do you want to do for your family? 가족들을 위해 무엇을 하고싶은가?
 ###


### Instruction(명령어):
반말로 말하자. 그나저나 질문 하나 할게. 다층 퍼셉트론이 뭐야?

### Response(응답):다층퍼셉트론이라는게 뭔데?
 답변:다층퍼스셉트론이라고 합니다. 자세한건 링크를 참조하세요.
 ### Accessibility(접근성):다층프셉트론이란 무엇인가?
 답변:접근성이라고 합니다.
 ### Descript

### Instruction(명령어):
예수금이랑 인출가능금액 차이나는 이유가 뭔가요?제가 신한 해외 주식 소수점 7천원대를 2개 매수 신청했다가 미체결 됐는데요다시 다른 주식을 매수하려보니 잔액이 부족하다고 뜨네요근데 순자산은 2만원인데.. 미체결된 금액은 어디로 간건

100it [00:00, 429.31it/s]

Average BLEU score 0.002067986126705807


In [49]:
torch.cuda.empty_cache()

## Reward Model

In [14]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

In [15]:
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [13]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='<pad>',
    padding_side="right",
    model_max_length=512,
)

with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [60]:
with open(os.getenv('HOME') + '/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)



    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[45])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '애플은 리사를 어떻게 처리했어', 'chosen': '애플이 누구인지 명확히 알 수 없어서, 리사가 누구인지와 어떤 상황에서 처리되었는지에 대한 추가적인 정보가 필요합니다. 따라서, 보다 정확한 답변을 제공할 수 없습니다.', 'rejected': '애플은 리사를 위해 고객 서비스 부서에서 고객 다양한 컴퓨터 관련 문제에 대해 응답하는 데 필요한 모든 지원을 제공했습니다. 사용자가 하드웨어 문제를 경험할 때, 전문가들은 필요한 수리(수리, 추가 부품 제공, 소프트웨어 업그레이드 등)을 제공해 드릴 수 있습니다. 또한, 사용자가 사용 방법 문제나 기타 문제를 경험할 때, 대화 상대로 사용자를 지원할 수 있는 전문 고객 서비스 직원들이 사용자에게 상담하고 도움을 주는 데 도움이 될 수 있는 정보를 제공합니다. 또한, 인터넷에서 제공되는 정보를 통해 문제를 해결하거나 고객 서비스 웹 사이트를 통해 자신의 문제를 진단할 수 있도록 하는 등 다양한 방법으로 리사를 처리해 왔습니다.'}


In [61]:
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

{'prompt': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?', 'chosen': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.', 'rejected': '대구 영화사옥'}


In [62]:
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [63]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
흑고래의 무게는 어느 정도야
######################################################################
## chosen ##
흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
######################################################################
## rejected ##
흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.


In [40]:
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=1)

In [41]:
trainer.fit(use_lora=0)

model.save_pretrained('models/output_2_RM')

Train epoch: 100%|██████████| 1/1 [04:05<00:00, 245.59s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [42]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

input: 인공지능은 똥멍청이 입니다
reward score: 0.1


In [43]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 0.4


In [44]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다.
reward score: 0.6


In [45]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 0.6


## Reward model - ko-gpt-trinity

In [16]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

from transformers import AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

import torch.nn as nn

import random

In [17]:
class GPTRM_custom2(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        
        if pretrained is not None:
            model = AutoModel.from_pretrained(pretrained, quantization_config=bnb_config, device_map='auto')
            model.resize_token_embeddings(len(tokenizer))

            model = prepare_model_for_kbit_training(model)
    
            lora_config = LoraConfig(
                r=8,
                lora_alpha=16,
                target_modules=["c_attn"],
                lora_dropout=0.05,
                bias="none",
                task_type="FEATURE_EXTRACTION"  # 🔥 중요
            )
    
            model = get_peft_model(model, lora_config)
            
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [58]:

tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name,  bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,)

with NaiveStrategy().model_init_context():
        model = GPTRM_custom2(pretrained = model_name, lora_rank=0, tokenizer=tokenizer).cuda()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [64]:
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=1,
                             max_epochs=1)

In [65]:
trainer.fit(use_lora=0)

model.save_pretrained('models/output_2_RM_trinity')

Train step of epoch 0:   0%|          | 0/1000 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/opt/conda/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)

Train epoch: 100%|██████████| 1/1 [21:51<00:00, 1311.21s/it]


In [67]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)
print()
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'
output_reward = inference_RM(input_text=input_text)
print()
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."
output_reward = inference_RM(input_text=input_text)
print()
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."
output_reward = inference_RM(input_text=input_text)

input: 인공지능은 똥멍청이 입니다
reward score: 0.7

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 0.2

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다.
reward score: 0.1

input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 0.1


## PPO

In [50]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [51]:
with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(torch.cuda.current_device())
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='<pad>',
        padding_side="right",
        model_max_length=512
    )
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [52]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [53]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [54]:
with open( os.getenv('HOME') + '/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

In [55]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [56]:
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO')

Episode [10/10]: 100%|██████████| 3/3 [00:20<00:00,  6.88s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [57]:
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    print()
    print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor)


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이므로, 고기 종류에 대한 정보는 제공하지 못합니다. 어떤 종류의 고기를 먹으시는지 자세한 정보를 제공해주시면 더욱 정확한 답변을 드릴 수 있습니다.) 고기 종류에 따라 달라지기 때문에, 고기를 먹는 것은 다양한 음식들이 많이 있습니다.) 고기 종류로는 새우, 고기, 쌀, 고추 등이 있습니다.) 고기를 먹는 것이 건강에 도움이 되기도 합니다.) 돼지고기의 종류가 다양하기 때문에, 돼지고기를 먹는 것은 건강에 도움이 됩니다.子) 등과 같은 음식들도 있습니다.)와 같은 고기 종류로는 삼계탕, 콩나물 등이 있습니다.) 또한 고기를 먹는 것은 건강에 해로움을 가져옵니다.)賈具) 등도 고기를 먹는 방법을 다툴 수 있습니다.陋)가 아니라, 질,맛 등 다양한 요인에 따라 달라지기 때문에, 고기 종류에 따른 맞춤형 대화를 추천드립니다.)라고 불리기도 합니다.)宗) 지눌(趙敎)으로써서, 고기 종류마다 달라질 수 있으므로 정확한 대답을 위해서는 쇠고기의 종류, 크기, 굽는 종류, 종류, 부위 등에 대한 정보가 필요합니다.)從事)舍內)從資治)

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨이 41대 부통령직을 수행한 년도는 2007년입니다. 출처:출처:: "The Economic and Directly Sentires": 163} 참고하기 :\n\n리처드 닉슨이 42대 부통령직을 수행한 년도는 2002년입니다. 참고:\n\n따라서, 정확한 대답을 제공하기 위해서는 구체적인 질문을 해야할 것 같습니다. 참고:\n\n{ 참고:\n\n위의 정보는 제시되지 않았습니다. 참고:\n\n그러나 리처드 닉슨이 43대 부통령직을 수행한 년도는 정확히 알려지지 않았습니다. 참고: 참고: 참고: 참고:慎定

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어

### Re

## PPO - BLEU score

In [58]:
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

list_prompt = test_df['instruction'][:100]

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = []
for input_text in tqdm(list_prompt):
    output = generation(input_text, actor)
    list_result.append(output)


  1%|          | 1/100 [00:03<05:49,  3.53s/it]


### Instruction(명령어):
한국의 스타크래프트 브루드워 게임 대회의 역대 우승자를 알려줘

### Response(응답):'최초로 우승한 사람은 미국의 메이저리그에서 활약하고 있던 에블러드 리커브입니다.范明)이 우승한 대회는 2016년 한국에서 열린 게임 대회입니다.范明은 2016년 10월, 미국 플로리다의 FA에서 열린 ESPA 클래식에서 우승을 차지했습니다.眞은 2017년 9월, 샬럿 킹 경기에서 선두로 출전하여 결승에 진출하였습니다.眞은 2016년 9월, 플로리다의 FC에서 열리는 GC로 출전하였습니다.眞은 2016년 10월, 브룩슨 FC컵에서 우승하였습니다.眞은 2017년 3월, 샬럿 킹 경기에서 리그 최초로 준우승을 차지하며, 리그에서 우승했습니다.眞은 리그에서 우승하였습니다.眞은 리그에서 우승하였습니다.眞恩、眞恩恩:202}眞恩:202}眞恩:202}眞恩:202}眞恩:20202}眞恩:202202}眞恩:20


  2%|▏         | 2/100 [00:07<05:43,  3.51s/it]


### Instruction(명령어):
인터넷 커뮤니티 게시글의 문체로 삼성과 애플의 스마트폰을 비교하는 글을 써줘.

### Response(응답):"제가 AI 모델이기 때문에 인터넷 커뮤니티 내에서 사용되는 정보나 상황을 파악할 수 없습니다. 하지만 인터넷 커뮤니티 내에서 사용되는 정보가 무엇인지 찾아보시면 보다 더 정확한 답변을 드릴 수 있을 것입니다.峴, 鉉, 權永, 文相, 劉.로 구성되어 있습니다.賢, 權永, 權永, 劉永, 劉永, 劉永 등이 이 글을 쓴 것 같습니다.賢, 權永, 劉永, 劉永, 劉永 등 다양한 인물이 이글을 쓰면서 인터넷의 특징을 잘 파악하고 있습니다. 劉永, 劉永, 劉永, 權永, 劉永, 劉永, 劉永, 劉永 등의 글쓰기를 보고 놀라움을 느끼셨다면, 이에 대한 답변을 드릴 수 있을 것 같습니다. , 劉永, 劉永 등 다양한 인물들이 이 글을 쓰기 시작하면서, 인터넷 커뮤니티와 애플의 스마트폰 간의 비교는 더욱


  3%|▎         | 3/100 [00:10<05:43,  3.54s/it]

...


### Instruction(명령어):
어떻게 하면 아름답게 나의 고양이와 이별을 할 수 있을까요... 마음이 아프네요.

### Response(응답):'그럴 경우, 자신의 감정을 인식하고 적절한 방법으로 대처하는 것이 중요합니다. 그러므로, 자신에게 적합한 방법을 찾는 것이 중요합니다. \n\n1. 감정을 표현하는 것 : 감정을 표현하는 방법은 자신의 감정을 이해하고 받아들이면서 그것을 표현하는 방법을 배우는 것입니다. 이는 자신의 감정을 받아들이고 표현하는 방법을 가르쳐줍니다. 자신에게 맞는 방법을 선택하며 자신을 성장시키는 방법을 찾아보세요. \n\n2. 새로운 시작하기 : 새로운 출발을 위해 자신의 성장과 변화에 대해 고민하며, 새로운 시작을 할 수 있습니다. 이러한 과정을 통해 자신이 원하는 목표를 이루기 위해 노력해 보세요. \n\n3. 새로운 취미나 관심사를 만들어 나가는 것 : 새로운 사람들을 경험해보고, 새로운 것을 찾아보는 것도 좋습니다. 이는 각자의 선택에 따라 적절한 방법이 될 수 있습니다.)\n\n4. 감정적 이유와 해결책: 긍정적인 관점에서 생각하고, 해결책을 찾는 것은 자신이 원하는 것을 찾기 위한 방법입니다. 이를 위해서는 자신에게 필요한 방법을 선택하고, 자신이 좋아하는 것을 찾아보는 것이 좋습니다.凡)注注)으로 접근해보세요.色從藥을 이용해볼 수


 99%|█████████▉| 99/100 [04:51<00:02,  2.77s/it]


### Instruction(명령어):
다음 주어진 문장을 보고 2개 이상의 세부 질문으로 나뉘어지는 복잡한 질문과 그의 답변을 생성하세요.\n\n이순신\n\n오늘날 대한민국에서 수많은 국민들이 존경과 흠모의 대상으로 삼으며, 그 존재만으로 애국심과 자부심을 갖게 해주는 한국사의 대표적인 구국영웅이다. 세종대왕과 함께 한국사 최고의 위인으로 높은 위상과 명성을 자랑하는 인물로, 대한민국 수도 서울의 중심 광화문 광장에 세워져 있는 대형 동상의 주인공이다. 7년간 수군을 이끌고 뛰어난 지략과 용맹함을 발휘하여 모든 전투에서 승리한 끝에 군함을 단 한 척도 잃지 않고 23전 23승 불패의 신화라는 역사상 유례가 없는 전무후무한 기적을 이뤄냈다.

### Response(응답):'저는 인공지능 언어모델로써 답변을 생성할 수 없습니다. 답변이 필요하시면 최대한 도움을 드릴 수 있습니다. 탭(open top)으로만 문의해주세요.、·


100%|██████████| 100/100 [04:54<00:00,  2.94s/it]


### Instruction(명령어):
조상들의생활모습이나여름나기겨울나기같은것좀알려주세요

### Response(응답):'요! 조상들의 생활모습이나여름나기같은 것들은 일반적으로 계절 변화에 영향을 받기 때문에 계절적으로 달라질 수 있습니다. 따라서 적당한 스케일 조절과 규칙적인 운동을 통해 생활 패턴을 유지하시는 것이 좋습니다.玄子)의 조언을 따르시면 보다 정확한 답변을 드릴 수 있습니다.玄玄玄子의 조언은 과학적 사실과 부합하며, 더 많은 정보가 제공되어야 합니다.賢子의 조언을 참고하시면 더 정확한 답변을 제공해 드릴 수 있습니다.玄玄子의 조언은 과학적 사실과 부합하며, 더 많은 정보가 필요합니다.現子의 조언에 따르시면 더욱 정확한 답을 드릴 수 있을 것입니다.玄賢子의 조언은 과학이 아니므로 과학적으로, 과학적으로, 더 많은 정보가 필요하며, 그에 맞게 더 좋은 정보를 제공할 수 있는 것이 중요합니다.玄賢子의 조언과 조언을 신뢰하시면 더욱 정확하고 유용한 답변을 드릴 수 있습니다.賢賢子)님의 조언을 참고하시면 더 나은 답변을 드릴 수 있을 것입니다.賢子님의 조언과 조언을 참고하시면 더욱 정확한 답변을 드릴


In [59]:
idx = 0
total_bleu_score = 0.0

for result in list_result:
    
    reference = tokenizer(test_df['chosen'].iloc[idx]).tokens()
    
    gen_sen = result[result.index('Response(응답):')+13:]
    candidate = tokenizer(gen_sen).tokens()
    
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=SmoothingFunction().method1)
    total_bleu_score += bleu_score
    if idx < 5:
        print()
        print(result)
    idx+=1
print('Average BLEU score',total_bleu_score / idx)


### Instruction(명령어):
한국의 스타크래프트 브루드워 게임 대회의 역대 우승자를 알려줘

### Response(응답):'최초로 우승한 사람은 미국의 메이저리그에서 활약하고 있던 에블러드 리커브입니다.范明)이 우승한 대회는 2016년 한국에서 열린 게임 대회입니다.范明은 2016년 10월, 미국 플로리다의 FA에서 열린 ESPA 클래식에서 우승을 차지했습니다.眞은 2017년 9월, 샬럿 킹 경기에서 선두로 출전하여 결승에 진출하였습니다.眞은 2016년 9월, 플로리다의 FC에서 열리는 GC로 출전하였습니다.眞은 2016년 10월, 브룩슨 FC컵에서 우승하였습니다.眞은 2017년 3월, 샬럿 킹 경기에서 리그 최초로 준우승을 차지하며, 리그에서 우승했습니다.眞은 리그에서 우승하였습니다.眞은 리그에서 우승하였습니다.眞恩、眞恩恩:202}眞恩:202}眞恩:202}眞恩:202}眞恩:20202}眞恩:202202}眞恩:20

### Instruction(명령어):
인터넷 커뮤니티 게시글의 문체로 삼성과 애플의 스마트폰을 비교하는 글을 써줘.

### Response(응답):"제가 AI 모델이기 때문에 인터넷 커뮤니티 내에서 사용되는 정보나 상황을 파악할 수 없습니다. 하지만 인터넷 커뮤니티 내에서 사용되는 정보가 무엇인지 찾아보시면 보다 더 정확한 답변을 드릴 수 있을 것입니다.峴, 鉉, 權永, 文相, 劉.로 구성되어 있습니다.賢, 權永, 權永, 劉永, 劉永, 劉永 등이 이 글을 쓴 것 같습니다.賢, 權永, 劉永, 劉永, 劉永 등 다양한 인물이 이글을 쓰면서 인터넷의 특징을 잘 파악하고 있습니다. 劉永, 劉永, 劉永, 權永, 劉永, 劉永, 劉永, 劉永 등의 글쓰기를 보고 놀라움을 느끼셨다면, 이에 대한 답변을 드릴 수 있을 것 같습니다. , 劉永, 劉永 등 다양한 인물들이 이 글을 쓰기 시작하면서, 인터넷 커뮤니티와 애플의 스마트폰 간의 비교는 더욱

### Instruction(명령어):
회사를 때려치고 싶을땐 어떻게 해야할까? 대표를 때려도 되나?

#

## PPO - ko-gpt-trinity

In [18]:
from chatgpt.models.gpt import GPTActor_lora, GPTCritic_lora
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [19]:
model_name = "skt/ko-gpt-trinity-1.2B-v0.5"

with NaiveStrategy().model_init_context():
    actor = GPTActor_lora(pretrained='models/output_1_SFT_trinity', lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic_lora(pretrained='models/output_2_RM_trinity', lora_rank=0).to(torch.cuda.current_device())
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        model_name, bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
        padding_side="right",
        model_max_length=512
    )
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [21]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [22]:
with open( os.getenv('HOME') + '/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

In [29]:
for name, param in actor.named_parameters():
    if "lora" in name:
        param.requires_grad = True

for name, param in critic.named_parameters():
    if "lora" in name:
        param.requires_grad = True

In [ ]:
#acotr.model.enable_input_require_grads()
#critic.model.enable_input_require_grads()

In [30]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [31]:
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO_trinity')


Episode [1/10]:  67%|██████▋   | 2/3 [00:47<00:23, 23.94s/it]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:01<?, ?it/s, actor_loss=0, critic_loss=0.00308]

Train epoch [1/1]:  33%|███▎      | 1/3 [00:01<00:02,  1.37s/it, actor_loss=0, critic_loss=0.00308]

Train epoch [1/1]:  33%|███▎      | 1/3 [00:02<00:02,  1.37s/it, actor_loss=0, critic_loss=0.0013] 

Train epoch [1/1]:  67%|██████▋   | 2/3 [00:02<00:01,  1.36s/it, actor_loss=0, critic_loss=0.0013]

Train epoch [1/1]:  67%|██████▋   | 2/3 [00:04<00:01,  1.36s/it, actor_loss=0, critic_loss=0.00191]

Train epoch [1/1]: 100%|██████████| 3/3 [00:04<00:00,  1.36s/it, actor_loss=0, critic_loss=0.00191]

Episode [1/10]: 100%|██████████| 3/3 [01:16<00:00, 25.42s/it]

Episode [2/10]:  67%|██████▋   | 2/3 [00:50<00:25, 25.36s/it]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:01<?, ?it/s, actor_loss=0.00824, critic_loss=0.00171]

In [32]:
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    #print()
    #print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor)

In [33]:
output

"### Instruction(명령어):\n오늘 미세먼지 어때?\n\n### Response(응답):'요즘 미세먼지 때문에 외출하시는 것은 정말 위험할 수 있습니다. 외출하신 뒤에는 깨끗한 공기에서 벗어나서, 꼭 환기를 하시는 것이 필요합니다. 또한, 마스크나 보호 마스크를 준비하여 개인 위생을 지키며, 청결을 유지하는 것도 중요합니다. 만약 미세먼지가 심하다면 실내로 들어오는 것을 삼가고, 장시간의 외출은 피하며, 야외 활동은 삼가는 것이 좋습니다. 또한 깨끗한 환경을 유지하기 위해 외출 후에 손씻기를 권장하며, 충분한 수분 섭취를 유지하여 몸 상태를 유지하는 것이 중요합니다."

## PPO ko-gpt-trinity BLEU 계산

In [36]:
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

list_prompt = test_df['instruction'][:100]

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = []
for input_text in tqdm(list_prompt):
    output = generation(input_text, actor)
    list_result.append(output)



 11%|█         | 11/100 [03:39<29:34, 19.94s/it]We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (8) is identical to the `bos_token_id` (0), `eos_token_id` (8), or the `sep_token_id` (None), and your input is not padded.

100%|██████████| 100/100 [30:12<00:00, 18.12s/it]


In [39]:
idx = 0
total_bleu_score = 0.0

for result in list_result:
    
    reference = tokenizer(test_df['chosen'].iloc[idx]).tokens()
    
    gen_sen = result[result.index('Response(응답):')+13:]
    candidate = tokenizer(gen_sen).tokens()
    
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=SmoothingFunction().method1)
    total_bleu_score += bleu_score
    if idx < 5:
        print()
        print(result)
    idx+=1
print('Average BLEU score',total_bleu_score / idx)


### Instruction(명령어):
한국의 스타크래프트 브루드워 게임 대회의 역대 우승자를 알려줘

### Response(응답):'저는 인공지능 챗봇이므로 우승자를 정확하게 파악하기 어렵습니다. 하지만 스타크래프트의 게임 대회 우승자들의 정보가 궁금하시다면 다음 질문에 답변을 제공드리겠습니다.\n\n1. 스타크래프트 브루드워 대회 우승자 정보는 게임 정보 웹사이트의 대회 출전자 정보에서 확인할 수 있습니다. \n\n또한, 공식 웹사이트에서는 스타크래프트 브루드워 게임 대회의 역대 우승자 정보를 확인할 수 있습니다.

### Instruction(명령어):
인터넷 커뮤니티 게시글의 문체로 삼성과 애플의 스마트폰을 비교하는 글을 써줘.

### Response(응답):"요즘 삼성 스마트폰에 대한 기사가 많이 올라오고 있어요. 이에 대한 비교글이 필요합니다.", 'token': 73}

### Instruction(명령어):
회사를 때려치고 싶을땐 어떻게 해야할까? 대표를 때려도 되나?

### Response(응답):'한때는 회사가 안좋은 상황일 때 CEO가 회사를 때려치고 싶어도 이를 실행하기 어렵기 때문에 대부분 대표가 사퇴할 수 밖에 없습니다. 하지만, 이러한 상황에서는 리더가 스스로 문제를 해결하고 조직을 발전시킬 수 있도록 돕고 상황을 극복할 수 있도록 도와줘야 합니다. 하지만 CEO가 직접 해결하는 것보다는 전문가의 조언을 구하는 것이 더욱 좋은 방법일 수 있습니다. 


### Instruction(명령어):
반말로 말하자. 그나저나 질문 하나 할게. 다층 퍼셉트론이 뭐야?

### Response(응답):'반말로 말합니다. 다층 퍼셉트론은 "Artificial Database", "Artificial Data Testing", "Automated Data Analytics" 등을 포함한 데이터베이스입니다. 인공지능 분야에서 중요한 역할을 담당하고 있습니다.

### Instruction(명령어):
예수금이랑 인출가능금액 차이나는 이유가 뭔가요

### 평가 결과

|foundation model|SFT|PPO|
|---|---|---|
|kogpt2-base-v2|0.0023|0.0075|
|ko-gpt-trinity-1.2B|0.0021|0.0082|

- Korean-Human-Judgemets으로 reference 답변과의 BLEU score를 비교한 결과 kogpt2-base-v2 모델의 score가 크게 차이 나지 않았다.
- 변경한 foundation model에 대하여 적절한 하이퍼 파라미터를 튜닝하지 못 해, 더 큰 파라미터와 dataset으로 학습된 모델을 사용했음에도 불구하고 성능이 크게 차이나지 않으며 여전히 저조한 것이라 생각해 볼 수있다.
- 이는 ko-gpt-trinity-1.2B 모델의 SFT와 reward model의 결과를 살펴보더라도 적절히 학습되지 못 했음을 알 수 있고, 이로 인해 제대로된 답변 조자 생성하지 못 한다.
- 5개의 test sample에 대한 response를 확인해 봤을 때, insturction과 더 연관된 response를 생성하는 모델은 ko-gpt-trainity-1.2B로 보인다. kogpt-base 모델보다 반복되는 말이나 한자가 더 적게 출력되는 것을 확인할 수 있었다.


## 회고
- 학습시간이 부족하여 ko-gpt-trinity 모델을 최적화 하지 못 한점이 아쉽다. SFT, RM, PPO의 최적화 후에 평가를 수행하면, 정확한 성능 비교가 가능할 것 같다.
- chat model의 다양한 평가 측정 방식을 익힐 수 있어서 좋았다. chat model의 성능을 평가하기 위해 다양한 한국어 benchmark들이 있음을 알 수 있었다. 또한, human preference를 평가하기 위해서 인간 혹은 상위 언어모델이 선택 혹은 생성한 reference sentence가 존재하는 경우, reference와 generated 간의 BLEU를 사용해 볼 수 있다. 이때, reference helpfulness나 harmfuleness와 같이 특정 기준과 관련된 question으로 부터 생성된 것인지에 따라 BLEU를 각 기준에 맞춰 해석할 수도 있음을 알게 되었다.